# ML Surrogates for Chemical Processes with Gurobi-ML

This notebook illustrates the use of TensorFlow Keras and Gurobi-ML to produce an ML surrogate based on data from a chemical process flowsheet.

**Note:** This notebook is adapted from the [OMLT Auto-thermal Reformer example](https://github.com/cog-imperial/OMLT/blob/main/docs/notebooks/Thermal/auto-thermal-reformer.ipynb). The data is sourced from the OMLT repository.

There are several reasons to build surrogate models for complex processes, even when higher fidelity models already exist (e.g., reduce model size, improve convergence reliability, replace models with externally compiled code and make them fully-equation oriented).

## 1. Setup and Configuration

In [ ]:
import os
import urllib.request
import pandas as pd
import gurobipy as gp
import gurobipy_pandas as gppd
from tensorflow import keras
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import StandardScaler
from gurobi_ml import add_predictor_constr

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# --- Configuration ---
DATA_URL = "https://raw.githubusercontent.com/cog-imperial/OMLT/main/docs/notebooks/data/reformer.csv"
DATA_FILE = "reformer.csv"

INPUT_COLS = ["Bypass Fraction", "NG Steam Ratio"]
OUTPUT_COLS = [
    "Steam Flow",
    "Reformer Duty",
    "AR",
    "C2H6",
    "C3H8",
    "C4H10",
    "CH4",
    "CO",
    "CO2",
    "H2",
    "H2O",
    "N2",
]

# Neural Network Parameters
N_LAYERS = 4
N_NEURONS = 20
LEARNING_RATE = 0.01
EPOCHS = 20

## 2. Data Loading and Preprocessing

In [ ]:
def load_reformer_data(url, filename):
    if not os.path.exists(filename):
        print(f"Downloading {filename}...")
        urllib.request.urlretrieve(url, filename)
    return pd.read_csv(filename, usecols=INPUT_COLS + OUTPUT_COLS)


df = load_reformer_data(DATA_URL, DATA_FILE)
dfin, dfout = df[INPUT_COLS], df[OUTPUT_COLS]

# Scale data using Scikit-Learn
scaler_in = StandardScaler().fit(dfin)
scaler_out = StandardScaler().fit(dfout)

X_train = scaler_in.transform(dfin)
y_train = scaler_out.transform(dfout)

# Capture physical bounds for optimization
input_bounds = pd.DataFrame({"lb": dfin.min(), "ub": dfin.max()})

df.head()

## 3. Surrogate Model Training

In [ ]:
# Build parameterized Keras model
nn = Sequential(
    [
        keras.Input(shape=(len(INPUT_COLS),)),
        *[Dense(N_NEURONS, activation="sigmoid") for _ in range(N_LAYERS)],
        Dense(len(OUTPUT_COLS)),
    ],
    name=f"reformer_sigmoid_{N_LAYERS}_{N_NEURONS}",
)

nn.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss="mse")

# Show architecture
nn.summary()

In [ ]:
print(f"\nTraining {nn.name}...")
history = nn.fit(X_train, y_train, epochs=EPOCHS, validation_split=0.2, verbose=2)

print(f"\nFinal Training Loss:   {history.history['loss'][-1]:.6f}")
print(f"Final Validation Loss: {history.history['val_loss'][-1]:.6f}")

## 4. Optimization Problem

We seek to maximize Hydrogen concentration (`H2`) while keeping Nitrogen (`N2`) below 0.34.

In [ ]:
m = gp.Model("Reformer_Optimization")

# Variables in physical space (indexed by column names)
x = gppd.add_vars(m, input_bounds, lb="lb", ub="ub", name="x")
y = gppd.add_vars(m, dfout.columns, lb=-gp.GRB.INFINITY, name="y")

# Intermediate scaled variables
x_scaled = gppd.add_vars(m, x.index, lb=-gp.GRB.INFINITY, name="x_scaled")
y_scaled = gppd.add_vars(m, y.index, lb=-gp.GRB.INFINITY, name="y_scaled")

# Connect variables via Gurobi-ML pipeline
# Gurobi-ML supports gurobipy-pandas objects (Series/DataFrames)
add_predictor_constr(m, scaler_in, x, x_scaled)
add_predictor_constr(m, nn, x_scaled, y_scaled)
add_predictor_constr(m, scaler_out, y, y_scaled)

# Objective and constraints using column names
m.setObjective(y["H2"], gp.GRB.MAXIMIZE)
m.addConstr(y["N2"] <= 0.34)

m.optimize()

## 5. Results

In [ ]:
if m.status == gp.GRB.OPTIMAL:
    print("\nOptimal Operating Point:")
    print(x.gppd.X)

    print("\nPredicted Concentrations:")
    print(f"{'H2':>16}: {y['H2'].X:.4f} (Maximized)")
    print(f"{'N2':>16}: {y['N2'].X:.4f} (Constraint: <= 0.34)")